In [18]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import classification_report, accuracy_score

# 1. Load your dataset
df = pd.read_csv('synthetic_health_risk_dataset_1000.csv')

# 2. Select the Features relevant to these 3 diseases
# We use BMI and nutritional intake as the main drivers
X = df[['age', 'bmi', 'sugar_g', 'sodium_mg', 'fat_g', 'carbs_g', 'protein_g']]

# 3. Select the Targets
y = df[['diabetes_risk', 'hypertension_risk', 'obesity_risk']]

# 4. Split into Train (80%) and Test (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 5. Scale the data (Crucial for medical accuracy)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Data prepared and scaled successfully.")

Data prepared and scaled successfully.


In [19]:
# Initialize the base XGBoost model
base_model = XGBClassifier(
    n_estimators=200, 
    learning_rate=0.05, 
    max_depth=5, 
    use_label_encoder=False, 
    eval_metric='logloss'
)

# Wrap it in a MultiOutput Classifier
multi_model = MultiOutputClassifier(base_model)

# Train the model
multi_model.fit(X_train_scaled, y_train)

print("Model training for Diabetes, Hypertension, and Obesity complete.")

Model training for Diabetes, Hypertension, and Obesity complete.


C:\Users\DELL\AppData\Local\Programs\Python\Python312\Lib\site-packages\xgboost\training.py:199: UserWarning: [17:15:29] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
C:\Users\DELL\AppData\Local\Programs\Python\Python312\Lib\site-packages\xgboost\training.py:199: UserWarning: [17:15:29] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
C:\Users\DELL\AppData\Local\Programs\Python\Python312\Lib\site-packages\xgboost\training.py:199: UserWarning: [17:15:30] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [23]:
# Sample testing data representing 5 different patient profiles
sample_patients = [
    {
        "description": "High Risk Patient (Obese/Diabetes Profile)",
        "data": [[55, 34.5, 95, 3800, 110, 350, 60]] 
        # [age, bmi, sugar_g, sodium_mg, fat_g, carbs_g, protein_g]
    },
    {
        "description": "Fit Young Adult (Healthy Profile)",
        "data": [[24, 22.1, 25, 1500, 50, 200, 80]]
    },
    {
        "description": "Elderly with High Salt Intake (Hypertension Profile)",
        "data": [[70, 26.8, 40, 5500, 70, 250, 55]]
    },
    {
        "description": "Average Office Worker (Sedentary/Moderate Risk)",
        "data": [[38, 28.2, 65, 3200, 85, 300, 65]]
    },
    {
        "description": "Athlete (High Calorie/Low Risk)",
        "data": [[29, 23.5, 35, 2200, 90, 450, 140]]
    }
]

# Code to run all 5 through your model
print("--- BATCH TEST RESULTS ---")
for patient in sample_patients:
    # Scale the individual data
    scaled_input = scaler.transform(patient['data'])
    
    # Get prediction from your MultiOutput XGBoost model
    prediction = multi_model.predict(scaled_input)[0]
    
    print(f"\nTarget Profile: {patient['description']}")
    print(f"Prediction (D, H, O): {prediction}")
    # Logic: 1 = High Risk, 0 = Low Risk

--- BATCH TEST RESULTS ---

Target Profile: High Risk Patient (Obese/Diabetes Profile)
Prediction (D, H, O): [1 1 1]

Target Profile: Fit Young Adult (Healthy Profile)
Prediction (D, H, O): [0 0 0]

Target Profile: Elderly with High Salt Intake (Hypertension Profile)
Prediction (D, H, O): [0 1 0]

Target Profile: Average Office Worker (Sedentary/Moderate Risk)
Prediction (D, H, O): [1 1 0]

Target Profile: Athlete (High Calorie/Low Risk)
Prediction (D, H, O): [0 0 0]


C:\Users\DELL\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\DELL\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\DELL\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\DELL\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
C:\Users\DELL\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2691: UserWa

saving the brain

In [24]:
import joblib

# Save the trained model
joblib.dump(multi_model, 'nutriscan_risk_model.pkl')

# Save the scaler (ESSENTIAL: the app must scale data the same way)
joblib.dump(scaler, 'nutriscan_scaler.pkl')

print("Model and Scaler saved as .pkl files. These are your 'final' artifacts.")

Model and Scaler saved as .pkl files. These are your 'final' artifacts.
